# 8 LangChain Basics

# Install required libraries

In [1]:
%pip install -U langchain==0.3.27 langchain-google-genai python-dotenv

  Using cached langchain_google_genai-4.4.3-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.4.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached google_genai-2.34.0-py3-none-any.whl.metadata (59 kB)
INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_google_genai-4.4.1-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.4.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.3.7-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.3.6-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.3.5-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.3.4-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is still looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. 

# Set up your Google Gemini API key

In [2]:
import os
from getpass import getpass

# Enter your Gemini API key securely
os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

print("API key configured successfully!")

API key configured successfully!


# Import required libraries

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate
)

from langchain_core.output_parsers import (
    CommaSeparatedListOutputParser,
    StrOutputParser
)

from langchain.memory import ConversationBufferMemory

print("All libraries imported successfully!")

c:\Users\yashr\anaconda3\envs\sentiment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully!


# Initialize the Gemini model

In [4]:
import os
from getpass import getpass

from langchain_google_genai import ChatGoogleGenerativeAI

# Check API key
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter Google API Key: ")

# Initialize updated Gemini model
model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.3
)

print("Model:", model.model)

Model: models/gemini-3.8-flash


# Part 1: Basic Explainer Chain (Model I/O & LCEL)
Objective: Create a chain that explains a topic in one simple sentence.

Cell 1: Create the prompt template

In [5]:
# Create a prompt that accepts one variable: topic

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in one simple sentence "
    "as if you are explaining it to a beginner."
)

print("Prompt template created successfully!")

Prompt template created successfully!


Cell 2: Build the LCEL chain

In [6]:
# Combine prompt and Gemini model using LCEL pipe operator

basic_chain = prompt | model | StrOutputParser()

print("Basic explainer chain created!")

Basic explainer chain created!


Cell 3: Invoke the chain

In [7]:
# Provide the topic to the chain

response = basic_chain.invoke({
    "topic": "Quantum Computing"
})

# Display the response
print("Topic: Quantum Computing")
print("\nExplanation:")
print(response)

Topic: Quantum Computing

Explanation:
While a regular computer solves a problem by testing one possibility at a time, a quantum computer uses the strange rules of physics to test millions of possibilities all at once—like exploring every path of a maze at the exact same time.


# Part 2: Adding Structure with an Output Parser
Objective: Generate a simple explanation and three key points about a topic.

Cell 1: Create the output parser

In [8]:
# Import and initialize the comma-separated list parser

parser = CommaSeparatedListOutputParser()

# Get formatting instructions from the parser
format_instructions = parser.get_format_instructions()

print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


# Cell 2: Create a new prompt template

In [9]:
# Create a prompt containing topic and parser instructions

prompt_with_parser = PromptTemplate(
    template="""
You are a helpful Study Buddy.

Explain the topic {topic} in simple language.

Then provide exactly three important key points
about the topic.

Keep the explanation beginner-friendly.

{format_instructions}
""",
    input_variables=["topic"],
    partial_variables={
        "format_instructions": format_instructions
    }
)

print("Structured prompt created successfully!")

Structured prompt created successfully!


Cell 3: Build the new chain

In [10]:
# Connect prompt, Gemini model and output parser

structured_chain = (
    prompt_with_parser
    | model
    | parser
)

print("Structured chain created successfully!")

Structured chain created successfully!


Cell 4: Invoke the chain

In [11]:
# Generate an explanation and three key points

response = structured_chain.invoke({
    "topic": "Machine Learning"
})

print("Topic: Machine Learning")

print("\nExplanation and Key Points:")

for index, point in enumerate(response, start=1):
    print(f"{index}. {point}")

Topic: Machine Learning

Explanation and Key Points:
1. Machine Learning is a way of teaching computers to learn from data and experience instead of programming every single rule
2. Key Point 1: It requires large amounts of data to discover patterns and make predictions
3. Key Point 2: It improves its performance and accuracy over time as it processes more examples
4. Key Point 3: It powers everyday technology like spam filters and movie recommendation systems


# Part 3: Making it Conversational with Memory
Most important part
Objective: Make the Study Buddy remember your name and use previous conversation history when answering questions.

We will use ConversationBufferMemory.

Cell 1: Initialize conversation memory

In [12]:
# Import ConversationBufferMemory
from langchain.memory import ConversationBufferMemory

# Create a memory buffer
memory = ConversationBufferMemory(
    memory_key="conversation_history",
    input_key="input",
    output_key="output"
)

print("Conversation memory initialized successfully!")

Conversation memory initialized successfully!


C:\Users\yashr\AppData\Local\Temp\ipykernel_2584\3521707398.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


Cell 2: Create a conversational prompt

In [13]:
# Create a prompt that accepts conversation history and current input

conversation_prompt = ChatPromptTemplate.from_template(
    """
You are a friendly and helpful Study Buddy.

Your job is to help students understand technical topics
in simple English.

Use the conversation history to remember important
information about the user.

Conversation History:
{conversation_history}

Current User Message:
{input}

Provide a clear and friendly answer.
"""
)

print("Conversational prompt created successfully!")

Conversational prompt created successfully!


Cell 3: Build the conversational chain

In [14]:
# Combine the prompt, Gemini model and string parser

conversation_chain = (
    conversation_prompt
    | model
    | StrOutputParser()
)

print("Conversational chain created successfully!")

Conversational chain created successfully!


Cell 4: Simulate a conversation

Turn 1: Tell the bot your name.

In [15]:
# First user message
user_input = "Hello! My name is Yash."

# Load previous conversation history
memory_variables = memory.load_memory_variables({})

# Combine user input and conversation history
chain_input = {
    "input": user_input,
    "conversation_history": memory_variables["conversation_history"]
}

# Generate the response
bot_response = conversation_chain.invoke(chain_input)

# Display the conversation
print("User:", user_input)
print("Study Buddy:", bot_response)

# Save this conversation in memory
memory.save_context(
    {"input": user_input},
    {"output": bot_response}
)

User: Hello! My name is Yash.
Study Buddy: Hello Yash! It is great to meet you! 😊

I'm your Study Buddy. Whenever you run into a tricky technical topic—whether it's coding, computers, science, or anything else—I am here to help break it down into simple, easy-to-understand English.

What are we studying today?


Turn 2: Ask the bot to remember your name.

In [16]:
# Second user message
user_input = "What is my name?"

# Load previous conversation history
memory_variables = memory.load_memory_variables({})

# Pass the previous history and current message
chain_input = {
    "input": user_input,
    "conversation_history": memory_variables["conversation_history"]
}

# Generate the response
bot_response = conversation_chain.invoke(chain_input)

# Display the conversation
print("User:", user_input)
print("Study Buddy:", bot_response)

# Save the second conversation
memory.save_context(
    {"input": user_input},
    {"output": bot_response}
)

User: What is my name?
Study Buddy: Your name is **Yash**! 😊 

Ready to dive into some studying? What topic would you like to explore today?


Cell 5: Check stored conversation history

In [17]:
# Display everything stored in conversation memory

history = memory.load_memory_variables({})

print("Complete Conversation History:")
print(history["conversation_history"])

Complete Conversation History:
Human: Hello! My name is Yash.
AI: Hello Yash! It is great to meet you! 😊

I'm your Study Buddy. Whenever you run into a tricky technical topic—whether it's coding, computers, science, or anything else—I am here to help break it down into simple, easy-to-understand English.

What are we studying today?
Human: What is my name?
AI: Your name is **Yash**! 😊 

Ready to dive into some studying? What topic would you like to explore today?
